# Myopsia: Notebook 01 - Data Cleaning & Exploratory Data Analysis (EDA)
### Project: AI-Based Eye Disease Prediction and Clinical Decision Support System Using Non-Image Clinical Data

> **Academic & Clinical Disclaimer:** This dataset is 100% synthetically generated for college-level machine learning experimentation. It contains no real patient records or validated clinical evidence, and must not be used for actual diagnostic medical practice.

---
### Objectives of this Notebook:
1. **Data Hygiene & Schema Validation:** Audit data types, record count ($N=5,000$), and patient ID uniqueness.
2. **Missing-Value Profiling:** Characterize missing patterns in Cup-to-Disc Ratio (CDR) and Intraocular Pressure (IOP) with clinical rationale.
3. **Physiological Distribution Analysis:** Validate continuous parameters against biological limits (Age, IOP, Visual Acuity in LogMAR).
4. **Symptom Constellation & Co-Occurrence:** Generate symptom prevalence heatmaps across the 7 eye disease classes.
5. **Class Imbalance Assessment:** Profile the moderate imbalance across diagnostic targets.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visual styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 110

print("Imported core analysis packages.")


## 1. Load Raw Clinical Dataset

In [ ]:
df = pd.read_csv('../eye_disease_dataset.csv')
print(f"Dataset Dimensions: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(5)


## 2. Integrity and Hygiene Audit

In [ ]:
# Check patient ID uniqueness
unique_ids = df['patient_id'].nunique()
print(f"Unique Patient IDs: {unique_ids} / {len(df)}")
assert unique_ids == 5000, "Duplicate patient IDs detected!"

# Check duplicates across clinical attributes
feature_cols = [c for c in df.columns if c != 'patient_id']
n_dups = df.duplicated(subset=feature_cols).sum()
print(f"Duplicate Clinical Encounters: {n_dups}")


## 3. Missing Value Analysis & Clinical Rationale

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_summary = pd.DataFrame({'Missing_Count': missing, 'Missing_Pct': missing_pct})
print(missing_summary[missing_summary['Missing_Count'] > 0])


In [ ]:
# Missingness by disease class
missing_by_condition = df.groupby('eye_disease')[['intraocular_pressure_left', 'cup_to_disc_ratio_left']].apply(
    lambda x: x.isnull().mean() * 100
).round(1)

fig, ax = plt.subplots(figsize=(10, 4.5))
missing_by_condition.plot(kind='bar', ax=ax, colormap='Spectral')
ax.set_title('Measurement Omission Rate by Clinical Condition (%)', fontsize=12, fontweight='bold')
ax.set_ylabel('Percentage Missing (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.legend(['IOP Left (Contagion Avoidance)', 'CDR Left (Non-Dilated Triage)'])
plt.tight_layout()
plt.show()


## 4. Continuous Parameter Distributions

In [ ]:
num_cols = ['age', 'intraocular_pressure_left', 'visual_acuity_left', 'cup_to_disc_ratio_left']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(x='eye_disease', y=col, data=df, ax=axes[i], palette='Set2')
    axes[i].set_title(f'Distribution of {col}', fontweight='bold')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=35, ha='right')
    axes[i].set_xlabel('')

plt.tight_layout()
plt.show()


## 5. Symptom Prevalence Matrix across Diagnoses

In [ ]:
symptom_cols = [
    'blurred_vision', 'black_spots_floaters', 'eye_pain', 'eye_redness',
    'eye_itching', 'eye_dryness', 'excessive_watering', 'double_vision',
    'flashes_of_light', 'light_sensitivity', 'difficulty_seeing_at_night',
    'eye_swelling', 'headache', 'discharge_from_eye', 'foreign_body_sensation'
]

symptom_prev = df.groupby('eye_disease')[symptom_cols].mean() * 100

plt.figure(figsize=(13, 6))
sns.heatmap(symptom_prev, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': 'Prevalence (%)'})
plt.title('Symptom Prevalence Heatmap across Diagnostic Classes (%)', fontsize=13, fontweight='bold')
plt.xlabel('Reported Symptom')
plt.ylabel('Diagnostic Condition')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()


## 6. Target Class Distribution

In [ ]:
class_dist = df['eye_disease'].value_counts()
plt.figure(figsize=(9, 4.5))
sns.barplot(x=class_dist.index, y=class_dist.values, palette='viridis')
plt.title('Target Class Distribution (Moderate Real-World Imbalance)', fontsize=12, fontweight='bold')
plt.ylabel('Patient Encounters')
plt.xticks(rotation=30, ha='right')
for i, v in enumerate(class_dist.values):
    plt.text(i, v + 15, f"{v} ({v/50:.1f}%)", ha='center', fontsize=9)
plt.tight_layout()
plt.show()
